# Kashmiri Whisper Speech-to-Text (Transcribe) Fine-Tuning

This notebook fine-tunes **OpenAI Whisper** on authentic spoken Kashmiri for high-accuracy Speech-to-Text (`task="transcribe"`).

### Instructions:
1. **Accelerator**: In the right sidebar under **Notebook options**, select **GPU T4 x2** or **GPU P100**.
2. **Internet**: Toggle **Internet ON**.
3. **Attach Dataset**: Click **+ Add Input** and add the Kaggle dataset: `gopikrishnan2005/kashmiri-text-to-speech` (2,984 recordings with `textcorpus.csv`).
4. Click **Run All**.

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
!pip -q install 'transformers==4.48.3' 'peft==0.14.0' 'accelerate==1.3.0' 'datasets>=3.2' 'evaluate>=0.4' jiwer soundfile librosa

In [ ]:
# 2. Clone the Cipher Repository
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/cipher')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/sameerahmad86232/cipher.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
print('Repository ready at:', repo)

In [ ]:
# 3. Run Whisper Transcribe Fine-Tuning
import os, subprocess

env = os.environ.copy()
env['WHISPER_MODEL'] = 'openai/whisper-small'
env['NUM_EPOCHS'] = '10'
env['BATCH_SIZE'] = '8'
env['LEARNING_RATE'] = '1e-4'
env['OUTPUT_DIR'] = '/kaggle/working/kashmiri-whisper-transcribe-adapter'

# Optional: Set your Hugging Face token here if you want automatic upload
# env['HF_TOKEN'] = 'your_huggingface_write_token'
# env['HF_REPO_ID'] = 'Sameer0313/kashmiri-whisper-transcribe-lora'

subprocess.run(['python', 'scripts/finetune-whisper-transcribe.py'], cwd=str(repo), env=env, check=True)

In [ ]:
# 4. Audio Transcription Smoke Test & Inference Verification
import torch, librosa
from pathlib import Path
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

adapter_path = Path('/kaggle/working/kashmiri-whisper-transcribe-adapter')
base_id = 'openai/whisper-small'

print('Loading fine-tuned Whisper model...')
processor = WhisperProcessor.from_pretrained(str(adapter_path))
base_model = WhisperForConditionalGeneration.from_pretrained(base_id, torch_dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(base_model, str(adapter_path))
model.eval()

# Find a sample WAV from Kaggle input to transcribe
sample_wavs = list(Path('/kaggle/input').glob('**/*.wav'))
if sample_wavs:
    test_wav = sample_wavs[0]
    print(f'Testing on sample: {test_wav.name}')
    audio, sr = librosa.load(str(test_wav), sr=16000)
    inputs = processor.feature_extractor(audio, sampling_rate=16000, return_tensors='pt').input_features.to('cuda', dtype=torch.float16)
    
    with torch.no_grad():
        predicted_ids = model.generate(inputs, forced_decoder_ids=processor.get_decoder_prompt_ids(language='urdu', task='transcribe'))
    transcription = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    print('\n================ KASHMIRI TRANSCRIPTION ================')
    print(transcription)
    print('========================================================')
else:
    print('Attach a Kashmiri dataset to test real-time audio transcription!')

In [ ]:
# 5. Check Exported Adapter & Download Link
from pathlib import Path
zip_file = Path('/kaggle/working/kashmiri-whisper-transcribe-adapter.zip')
if zip_file.exists():
    print(f'Done! Download the adapter directly from the Kaggle Output panel:')
    print(f'File: {zip_file.name} ({zip_file.stat().st_size / 1024 / 1024:.2f} MB)')
else:
    print('Check /kaggle/working/ for generated files.')